## Optical Flow

**Optical flow** is the apparent motion of brightness patterns between two consecutive frames. For every point it tells us *where the point went*, i.e. a motion vector $(u, v)$. It lets us estimate motion without knowing anything about the objects in the scene.

The key assumption is **brightness constancy**: a point keeps its brightness while it moves,

$$I(x, y, t) = I(x + u,\, y + v,\, t + 1).$$

Using the first-order Taylor approximation of the right-hand side we get

$$I_x u + I_y v + I_t = 0,$$

where $I_x, I_y$ are the spatial image gradients and $I_t$ is the temporal gradient (difference between the frames). This is **one equation for two unknowns** $(u, v)$: from a single pixel we can only recover the motion component along the image gradient (the so-called *aperture problem*). Optical flow algorithms differ in how they get the missing equation.

There are two families:

* **Sparse** optical flow estimates the motion of a *selected set of points*. We use the **Lucas-Kanade** algorithm.
* **Dense** optical flow estimates a motion vector for *every pixel*. We use the **Farnebäck** algorithm.

In this notebook we try both on the same video.

In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]

## 1. Sparse optical flow: Lucas-Kanade

The [Lucas-Kanade algorithm](https://en.wikipedia.org/wiki/Lucas%E2%80%93Kanade_method) gets the missing equation by assuming that all pixels in a small window move **together**, i.e. share the same $(u, v)$. Every pixel in the window gives one equation and we solve the system by least squares. The matrix of this system,

$$\begin{bmatrix} \sum I_x^2 & \sum I_x I_y \\ \sum I_x I_y & \sum I_y^2 \end{bmatrix},$$

is exactly the matrix of the **Harris corner detector**. It can be inverted reliably only if the window contains gradients in two different directions, i.e. a **corner**. On a flat region or along a straight edge (the aperture problem again) the motion is ambiguous.

So Lucas-Kanade needs good points to track, and we find them with the **Shi-Tomasi** corner detector, an enhanced variant of the Harris detector. You can learn more in the original [paper](https://users.cs.duke.edu/~tomasi/papers/shi/TR_93-1399_Cornell.pdf) or in the very comprehensible OpenCV [tutorial](https://docs.opencv.org/3.4/d4/d8c/tutorial_py_shi_tomasi.html).

### Initialisation

We use `vtest.avi`, a sample video of the OpenCV repository: a **static camera** observing pedestrians who walk across the scene. [Download it](https://github.com/opencv/opencv/raw/4.x/samples/data/vtest.avi) and place it as `data/vtest.avi`. The video runs at 10 frames per second.

In [ ]:
video = cv2.VideoCapture('data/vtest.avi')

# Shi-Tomasi corner detection
config_st = {'maxCorners': 100,    # track at most this many points
             'qualityLevel': 0.3,  # keep only corners at least 30 % as good as the best one
             'minDistance': 7,     # minimum distance between two corners [px]
             'blockSize': 7}       # size of the window used to compute the corner response

# Lucas-Kanade optical flow
config_lk = {'winSize': (15, 15),  # window in which the motion is assumed to be constant
             'maxLevel': 2,        # number of pyramid levels (0 = no pyramid), allows larger motion
             'criteria': (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)}  # stop after 10 iterations or when converged

### Find Initial Keypoints to Track

We jump to a frame in which people are in the scene, convert it to grayscale (optical flow works on intensities) and find the corners. Most of them lie on the static background (buildings, lamp post, ...), and a few on the walking people.

In [ ]:
# Create some random colors, one per point
color = np.random.randint(0, 255, (100, 3))

# Jump to a frame with people in the scene and find keypoints
first = 100
video.set(cv2.CAP_PROP_POS_FRAMES, first)
ret, source = video.read()

src_gray = cv2.cvtColor(source, cv2.COLOR_BGR2GRAY)
p_src = cv2.goodFeaturesToTrack(src_gray, mask=None, **config_st)

keypoints = source.copy()
for i, p in enumerate(p_src):
    x, y = p.ravel()
    cv2.circle(keypoints, (int(x), int(y)), 5, color[i].tolist(), -1)
plt.imshow(cv2.cvtColor(keypoints, cv2.COLOR_BGR2RGB))

In order to better illustrate the optical flow detector at work, we will not apply it on consecutive frames. Instead, we "fast forward" the video by a few frames, so the perceived motion is larger.

In [ ]:
# Fast forward: the second frame is 3 frames (0.3 s) after the first one
for _ in range(2):
    ret, frame = video.read()

### Estimate Optical Flow

`cv2.calcOpticalFlowPyrLK` takes the two frames and the points in the first frame, and returns

* `p_dst`: where each point has moved to,
* `status`: 1 if the point was found in the second frame, 0 if it was lost,
* `err`: the matching error of each point.

We keep only the points that were successfully tracked.

In [ ]:
# Read next frame
ret, target = video.read()
dst_gray = cv2.cvtColor(target, cv2.COLOR_BGR2GRAY)

# Calculate optical flow
p_dst, status, err = cv2.calcOpticalFlowPyrLK(src_gray, dst_gray, p_src, None, **config_lk)

# Select points that have been successfully tracked
p_dst = p_dst[status==1]
p_src = p_src[status==1]

print(f'{len(p_dst)} points tracked')

Let's now draw the results. Every arrow starts at the position of the point in the first frame and ends at its position in the second frame, so the arrows show the motion.

In [ ]:
# Create an empty mask image for drawing purposes
mask = np.zeros_like(source)

# Draw the tracks
for i, (dst, src) in enumerate(zip(p_dst, p_src)):
    x_dst, y_dst = dst
    x_src, y_src = src

    mask = cv2.arrowedLine(mask, (int(x_src), int(y_src)), (int(x_dst), int(y_dst)),
                           color[i].tolist(), 2, tipLength=0.5)
    target = cv2.circle(target, (int(x_src), int(y_src)), 5, color[i].tolist(), -1)

result = cv2.add(target, mask)

In [ ]:
plt.subplot(121), plt.imshow(cv2.cvtColor(source, cv2.COLOR_BGR2RGB)), plt.title('Source')
plt.subplot(122), plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB)), plt.title('Target with the estimated motion')

In [ ]:
compound = cv2.addWeighted(result, 0.75, source, 0.25, gamma=0)
plt.imshow(cv2.cvtColor(compound, cv2.COLOR_BGR2RGB))

### Tracking points over a whole video

Two frames give us one motion vector per point. If we repeat the procedure for every pair of consecutive frames, always starting from the points found in the previous step, we **track the points through the video** and draw their trajectories.

Two details:

* Points that are lost (`status == 0`) are dropped. We drop their colors too, so every surviving point keeps its color.
* After each step the current frame and points become the "previous" ones for the next step.

In [ ]:
n_frames = 30

# Go back to the first frame and detect the points again
video.set(cv2.CAP_PROP_POS_FRAMES, first)
ret, frame = video.read()
prev_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
p_prev = cv2.goodFeaturesToTrack(prev_gray, mask=None, **config_st)
colors = np.random.randint(0, 255, (len(p_prev), 3))

trails = np.zeros_like(frame)
for _ in range(n_frames):
    ret, frame = video.read()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    p_next, status, err = cv2.calcOpticalFlowPyrLK(prev_gray, gray, p_prev, None, **config_lk)

    # Keep the successfully tracked points (and their colors)
    tracked = status.ravel() == 1
    p_next, p_prev, colors = p_next[tracked], p_prev[tracked], colors[tracked]

    for new, old, c in zip(p_next.reshape(-1, 2), p_prev.reshape(-1, 2), colors):
        trails = cv2.line(trails, (int(old[0]), int(old[1])), (int(new[0]), int(new[1])), c.tolist(), 2)
        frame = cv2.circle(frame, (int(new[0]), int(new[1])), 5, c.tolist(), -1)

    # Current frame and points become the previous ones
    prev_gray = gray
    p_prev = p_next.reshape(-1, 1, 2)

print(f'{len(p_prev)} of the initial points survived {n_frames} frames')
plt.imshow(cv2.cvtColor(cv2.add(frame, trails), cv2.COLOR_BGR2RGB))

## 2. Dense optical flow: Farnebäck

Lucas-Kanade gives us the motion of a few good points only. Often we would like to know how **every pixel** moves, for instance to find out which parts of the image are moving at all.

The [Farnebäck algorithm](https://docs.opencv.org/4.x/dc/d6b/group__video__track.html#ga5d10ebbd59fe09c5f650289ec0ece5af) approximates the neighbourhood of every pixel by a **quadratic polynomial**. If the neighbourhood moves, its polynomial changes in a predictable way, and the displacement can be computed from the change. As in Lucas-Kanade, an image pyramid helps to deal with larger motion.

The result is an array `flow` of the same size as the image with **two channels**: `flow[y, x] = (u, v)` is the displacement of the pixel at `(x, y)` between the two frames.

In [ ]:
config_fb = {'pyr_scale': 0.5,   # each pyramid level is half the size of the previous one
             'levels': 3,        # number of pyramid levels
             'winsize': 15,      # averaging window: larger = smoother and more robust, but blurrier flow
             'iterations': 3,    # iterations on each pyramid level
             'poly_n': 5,        # size of the neighbourhood used for the polynomial approximation
             'poly_sigma': 1.2,  # Gaussian std. dev. used to smooth the derivatives for the polynomial
             'flags': 0}

Dense methods work best when the motion between the frames is small, so this time we use two **consecutive** frames.

In [ ]:
video.set(cv2.CAP_PROP_POS_FRAMES, first)
ret, source = video.read()
ret, target = video.read()

src_gray = cv2.cvtColor(source, cv2.COLOR_BGR2GRAY)
dst_gray = cv2.cvtColor(target, cv2.COLOR_BGR2GRAY)

flow = cv2.calcOpticalFlowFarneback(src_gray, dst_gray, None, **config_fb)
print(flow.shape)

### Visualising the flow

A two-channel array is hard to look at. We turn every vector into **polar coordinates** (length and direction) and encode them as a color in the **HSV** color space:

* the **hue** is the direction of the motion,
* the **value** (brightness) is the speed. No motion is black.

In [ ]:
def flow_to_color(flow):
    magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])

    hsv = np.zeros((*flow.shape[:2], 3), dtype=np.uint8)
    hsv[..., 0] = angle * 180 / np.pi / 2    # OpenCV hue is in [0, 180) for 8-bit images
    hsv[..., 1] = 255
    hsv[..., 2] = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

To read the picture we need a legend: a **color wheel** shows which direction belongs to which color. We build it by running the same conversion on an artificial flow field in which every pixel moves away from the center.

In [ ]:
size = 200
u, v = np.meshgrid(np.linspace(-1, 1, size), np.linspace(-1, 1, size))
wheel = flow_to_color(np.dstack((u, v)).astype(np.float32))
wheel[np.hypot(u, v) > 1] = 0    # keep only the disk

plt.subplot(131), plt.imshow(cv2.cvtColor(source, cv2.COLOR_BGR2RGB)), plt.title('Source')
plt.subplot(132), plt.imshow(flow_to_color(flow)), plt.title('Dense optical flow')
plt.subplot(133), plt.imshow(wheel), plt.title('Color wheel')

Note that the brightness is **normalised** to the fastest pixel in this frame pair, so colors can only be compared within one picture, not between pictures.

### Vectors as arrows

To compare with Lucas-Kanade, we can also draw the dense flow as arrows. Drawing a vector for every pixel would be unreadable, so we draw one on a regular grid.

In [ ]:
step = 16     # distance between the arrows [px]
scale = 3     # make the arrows longer to see them better

arrows = target.copy()
for y in range(step // 2, flow.shape[0], step):
    for x in range(step // 2, flow.shape[1], step):
        u, v = flow[y, x]
        arrows = cv2.arrowedLine(arrows, (x, y), (int(x + scale * u), int(y + scale * v)),
                                 (0, 255, 0), 1, tipLength=0.4)
plt.imshow(cv2.cvtColor(arrows, cv2.COLOR_BGR2RGB))

### Motion mask

Since we know the speed of every pixel, a simple threshold on the length of the vectors tells us **which pixels are moving**. This works well when the camera is static, as in this video. The threshold is in pixels per frame; try different values.

In [ ]:
threshold = 1.0

magnitude = np.hypot(flow[..., 0], flow[..., 1])
moving = magnitude > threshold

plt.subplot(121), plt.imshow(moving, cmap='gray'), plt.title(f'Pixels moving faster than {threshold} px/frame')
plt.subplot(122), plt.imshow(cv2.cvtColor(target * moving[..., None].astype(np.uint8), cv2.COLOR_BGR2RGB)), plt.title('Moving parts of the frame')

## Sparse vs. dense

| | Lucas-Kanade (sparse) | Farnebäck (dense) |
|---|---|---|
| Output | motion of selected points | motion of every pixel |
| Needs | good corners to track | nothing |
| Speed | fast (a few points) | slower (every pixel) |
| Typical use | tracking points through a video, camera motion, as part of a tracker | motion segmentation, video analysis, frame interpolation |

Both rely on brightness constancy and small motion, so both struggle with fast motion, lighting changes and textureless regions.